In [ ]:
import sys
from google.colab import drive

drive.mount('/content/gdrive')
base_dir = "/content/gdrive/MyDrive/CS_5782_Final_Project"
sys.path.append(base_dir)

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [ ]:
from torch_geometric.datasets import Planetoid
import torch
import torch.nn.functional as F

dataset = Planetoid(root='/tmp/Cora', name='Cora')
data = dataset[0]

print(data)
print("x shape:", data.x.shape)
print("edge_index shape:", data.edge_index.shape)
print("num classes:", dataset.num_classes)

Processing...


Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])
x shape: torch.Size([2708, 1433])
edge_index shape: torch.Size([2, 10556])
num classes: 7


Done!


In [ ]:
import model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data = data.to(device)

sgc_model = model.SGC(
    in_channels=dataset.num_features,
    out_channels=dataset.num_classes,
    K=2,
    cached=True,
    dropout=0.0
).to(device)

logits = sgc_model(data.x, data.edge_index)

print("logits shape:", logits.shape)

logits shape: torch.Size([2708, 7])


In [ ]:
loss = F.cross_entropy(logits[data.train_mask], data.y[data.train_mask])
print("loss:", loss.item())

loss.backward()
print("backward success")

loss: 1.9486736059188843
backward success


In [ ]:
sgc_model = model.SGC(
    in_channels=dataset.num_features,
    out_channels=dataset.num_classes,
    K=2,
    cached=True,
    dropout=0.0
).to(device)

optimizer = torch.optim.Adam(sgc_model.parameters(), lr=0.2, weight_decay=5e-6)

def evaluate(model, data):
    model.eval()
    with torch.no_grad():
        logits = model(data.x, data.edge_index)
        pred = logits.argmax(dim=1)

        accs = []
        for mask in [data.train_mask, data.val_mask, data.test_mask]:
            correct = (pred[mask] == data.y[mask]).sum().item()
            acc = correct / int(mask.sum())
            accs.append(acc)
    return accs

best_val = 0.0
best_test = 0.0
best_epoch = 0

for epoch in range(1, 101):
    sgc_model.train()
    optimizer.zero_grad()

    logits = sgc_model(data.x, data.edge_index)
    loss = F.cross_entropy(logits[data.train_mask], data.y[data.train_mask])

    loss.backward()
    optimizer.step()

    train_acc, val_acc, test_acc = evaluate(sgc_model, data)

    if val_acc > best_val:
        best_val = val_acc
        best_test = test_acc
        best_epoch = epoch

    if epoch % 10 == 0:
        print(
            f"Epoch {epoch:03d} | "
            f"Loss {loss.item():.4f} | "
            f"Train {train_acc:.4f} | "
            f"Val {val_acc:.4f} | "
            f"Test {test_acc:.4f}"
        )

print("Best epoch:", best_epoch)
print("Best val:", best_val)
print("Test at best val:", best_test)

Epoch 010 | Loss 0.0035 | Train 1.0000 | Val 0.7460 | Test 0.7670
Epoch 020 | Loss 0.0002 | Train 1.0000 | Val 0.7520 | Test 0.7720
Epoch 030 | Loss 0.0001 | Train 1.0000 | Val 0.7620 | Test 0.7720
Epoch 040 | Loss 0.0001 | Train 1.0000 | Val 0.7680 | Test 0.7760
Epoch 050 | Loss 0.0001 | Train 1.0000 | Val 0.7680 | Test 0.7770
Epoch 060 | Loss 0.0001 | Train 1.0000 | Val 0.7720 | Test 0.7730
Epoch 070 | Loss 0.0002 | Train 1.0000 | Val 0.7740 | Test 0.7750
Epoch 080 | Loss 0.0002 | Train 1.0000 | Val 0.7720 | Test 0.7780
Epoch 090 | Loss 0.0003 | Train 1.0000 | Val 0.7740 | Test 0.7800
Epoch 100 | Loss 0.0004 | Train 1.0000 | Val 0.7800 | Test 0.7830
Best epoch: 96
Best val: 0.782
Test at best val: 0.783


In [ ]:
sgc_model = model.SGC(
    in_channels=dataset.num_features,
    out_channels=dataset.num_classes,
    K=2,
    cached=True,
    dropout=0.0
).to(device)

print(sgc_model._cached_x is None)   # True

_ = sgc_model(data.x, data.edge_index)

print(sgc_model._cached_x is None)   # False
print(sgc_model._cached_x.shape)

True
False
torch.Size([2708, 1433])


In [ ]:
model1 = model.SGC(dataset.num_features, dataset.num_classes, K=2, cached=False).to(device)
model2 = model.SGC(dataset.num_features, dataset.num_classes, K=2, cached=True).to(device)

model2.load_state_dict(model1.state_dict())

with torch.no_grad():
    out1 = model1(data.x, data.edge_index)
    out2 = model2(data.x, data.edge_index)

print(torch.allclose(out1, out2, atol=1e-6))

True
